In [2]:
import sys
sys.path.append("..")

import numpy as np
from src.data import load_data
from src import eda
from src.preprocessing import clean_data, split_data, build_preprocessor

In [ ]:
df_raw = load_data()
df = clean_data(df_raw)

print("Rows before:", len(df_raw), "| after:", len(df))
print("Duplicates in raw data:", eda.count_duplicates(df_raw))
eda.check_categories(df)  

Rows before: 2000 | after: 2000
Duplicates in raw data: 0
Gender: unexpected values = None
Physical_Activity: unexpected values = None
Part_Time_Job: unexpected values = None
Diet_Quality: unexpected values = None
Previous_Mental_Health_History: unexpected values = None
Mental_Health_Risk: unexpected values = None


In [4]:
X_train, X_test, y_train, y_test = split_data(df)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("\nTrain class proportions:")
print(y_train.value_counts(normalize=True).round(2))
print("\nTest class proportions:")
print(y_test.value_counts(normalize=True).round(2))

Train: (1600, 15) | Test: (400, 15)

Train class proportions:
Mental_Health_Risk
2    0.34
1    0.33
0    0.33
Name: proportion, dtype: float64

Test class proportions:
Mental_Health_Risk
2    0.34
1    0.33
0    0.33
Name: proportion, dtype: float64


In [5]:
print("Train:")
print(eda.missing_summary(X_train))
print("\nTest:")
print(eda.missing_summary(X_test))

Train:
                    missing values  percentage
Sleep_Hours                     49      3.0625
GPA                             27      1.6875
Social_Media_Hours              49      3.0625
Financial_Stress                71      4.4375
Family_Support                  40      2.5000
Attendance_Rate                 34      2.1250
Diet_Quality                    53      3.3125

Test:
                    missing values  percentage
Sleep_Hours                     13        3.25
GPA                             11        2.75
Social_Media_Hours               9        2.25
Financial_Stress                20        5.00
Family_Support                  14        3.50
Attendance_Rate                 13        3.25
Diet_Quality                    16        4.00


In [6]:
numeric_cols = X_train.select_dtypes(include="number").columns.to_list()
eda.detect_outliers_iqr(X_train, numeric_cols)

,column,lower,upper,n_outliers
0,Age,17.000,25.000,38
1,Year_of_Study,-3.500,8.500,0
2,Sleep_Hours,3.750,9.750,8
3,Study_Hours_Per_Week,3.600,36.400,4
4,GPA,1.640,3.480,6
5,Social_Media_Hours,3.125,8.925,17
6,Screen_Time_Hours,3.950,13.150,6
7,Financial_Stress,0.000,12.000,0
8,Family_Support,1.000,12.200,0
9,Attendance_Rate,66.250,104.250,8


In [7]:
from src.preprocessing import ORDINAL_COLS

ordinal_cols = list(ORDINAL_COLS.keys())
nominal_cols = [c for c in X_train.select_dtypes(include="object").columns
                if c not in ordinal_cols]

print("Ordinal  :", ordinal_cols)
print("Nominal  :", nominal_cols)

Ordinal  : ['Physical_Activity', 'Diet_Quality']
Nominal  : ['Gender', 'Part_Time_Job', 'Previous_Mental_Health_History']


C:\Users\USER\AppData\Local\Temp\ipykernel_18712\3949542276.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  nominal_cols = [c for c in X_train.select_dtypes(include="object").columns


In [ ]:
counts = y_train.value_counts()
ratio = counts.max() / counts.min()

print(counts.to_dict())
print("Imbalance ratio:", round(ratio, 2))

USE_SMOTE = bool(ratio > 1.5)  
print("USE_SMOTE =", USE_SMOTE)

{2: 544, 1: 528, 0: 528}
Imbalance ratio: 1.03
USE_SMOTE = False


In [ ]:
preprocessor = build_preprocessor(X_train)

X_train_prep = preprocessor.fit_transform(X_train)  
X_test_prep = preprocessor.transform(X_test)       

print("Train:", X_train_prep.shape, "| Test:", X_test_prep.shape)
print("NaN train:", np.isnan(X_train_prep).sum(), "| NaN test:", np.isnan(X_test_prep).sum())
print()
print(preprocessor.get_feature_names_out())

Train: (1600, 19) | Test: (400, 19)
NaN train: 0 | NaN test: 0

['numeric__Age' 'numeric__Year_of_Study' 'numeric__Sleep_Hours'
 'numeric__Study_Hours_Per_Week' 'numeric__GPA'
 'numeric__Social_Media_Hours' 'numeric__Screen_Time_Hours'
 'numeric__Financial_Stress' 'numeric__Family_Support'
 'numeric__Attendance_Rate' 'ordinal__Physical_Activity'
 'ordinal__Diet_Quality' 'nominal__Gender_Female' 'nominal__Gender_Male'
 'nominal__Gender_Non-binary' 'nominal__Part_Time_Job_No'
 'nominal__Part_Time_Job_Yes' 'nominal__Previous_Mental_Health_History_No'
 'nominal__Previous_Mental_Health_History_Yes']
